# Investment Profile Router — Conditional LangGraph Workflow

## Architecture
```
                                    ┌──→ conservative_advisor ──→┐
                                    │    (LLM: bonds, FDs, gold)  │
                                    │                             │
START ──→ profile_analyzer ──→ router ──→ moderate_advisor ──────┼──→ report_generator ──→ END
              (LLM)          (conditional     (LLM: balanced      │        (LLM)
          assess risk         edge, NO LLM)   funds, blue chip)   │
          appetite)                │                              │
                                    └──→ aggressive_advisor ──────┘
                                         (LLM: equity, small cap)
```

## Key Concept: Conditional Edge vs Parallel Edge
| Parallel (previous project) | Conditional (this project) |
|---|---|
| ALL paths execute | ONLY ONE path executes |
| Fan-out then fan-in | Branch then converge |
| Used for simultaneous evaluation | Used for routing based on decision |

## LLM Call Count: Exactly 3
1. profile_analyzer — assess risk appetite
2. One of: conservative / moderate / aggressive advisor
3. report_generator — final investor report

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import json

load_dotenv()
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## State Schema

All fields declared upfront — input, intermediate, and final output.

In [2]:
class InvestorState(TypedDict):
    # ── Input fields (provided by caller) ──────────────────────────
    investor_name: str
    age: int                        # younger = can take more risk
    monthly_income: float           # INR
    monthly_savings: float          # amount available to invest
    investment_horizon_years: int   # how long they can stay invested
    existing_liabilities: str       # home loan, car loan, none etc.
    investment_goal: str            # retirement, wealth creation, child education etc.
    loss_tolerance: str             # how they feel about losing money: panic / accept / embrace

    # ── profile_analyzer output ─────────────────────────────────────
    risk_category: str              # CONSERVATIVE / MODERATE / AGGRESSIVE
    risk_score: int                 # 1-10
    risk_reasoning: str             # why this category

    # ── advisor output (only ONE of these will be populated) ────────
    recommended_instruments: List[str]   # list of instruments
    allocation_breakdown: str            # e.g. 60% equity, 40% debt
    advisor_rationale: str               # why these instruments

    # ── report_generator output ─────────────────────────────────────
    final_report: str               # full investor-facing report

## Node 1 — Profile Analyzer (LLM)

Reads the full investor profile and assigns a risk category.
This is the ONLY node that runs before the routing decision.

In [3]:
def profile_analyzer(state: InvestorState) -> dict:
    """
    Node 1: Profile Analyzer (LLM)
    Reads investor profile holistically.
    Assigns: risk_category, risk_score, risk_reasoning.
    This output drives the conditional routing decision.
    """
    prompt = f"""
You are a SEBI-registered investment advisor doing an initial risk profiling.

Investor Profile:
  Name                   : {state['investor_name']}
  Age                    : {state['age']}
  Monthly income         : ₹{state['monthly_income']:,.0f}
  Monthly savings        : ₹{state['monthly_savings']:,.0f}
  Investment horizon     : {state['investment_horizon_years']} years
  Existing liabilities   : {state['existing_liabilities']}
  Investment goal        : {state['investment_goal']}
  Loss tolerance         : {state['loss_tolerance']}

Categorise this investor's risk appetite.

Guidelines:
  CONSERVATIVE : age 50+, short horizon, high liabilities, panics at loss
  MODERATE     : balanced profile, medium horizon, accepts some loss
  AGGRESSIVE   : young, long horizon, low liabilities, embraces volatility

Give a risk score 1-10 (1=most conservative, 10=most aggressive).
Give a one-line reasoning.

Respond ONLY in raw JSON. No markdown, no extra text:
{{"risk_category": "<CONSERVATIVE/MODERATE/AGGRESSIVE>",
  "risk_score": <number>,
  "risk_reasoning": "<one line>"}}
"""
    response = model.invoke(prompt)
    result = json.loads(response.content)
    print(f"[Profile Analyzer] → Category: {result['risk_category']}, Score: {result['risk_score']}")
    print(f"[Profile Analyzer] → Reasoning: {result['risk_reasoning']}")
    return {
        "risk_category": result["risk_category"],
        "risk_score": result["risk_score"],
        "risk_reasoning": result["risk_reasoning"]
    }

## Router — Conditional Edge (NO LLM)

This is NOT a node. It is a routing function attached to a conditional edge.
It reads risk_category from state and returns the name of the next node.
Pure Python logic — no LLM, no ambiguity.

In [4]:
def route_by_risk(state: InvestorState) -> str:
    """
    Conditional Edge Router — NO LLM.
    Reads risk_category set by profile_analyzer.
    Returns the name of the next node to execute.
    Only ONE advisor node will run per workflow execution.
    """
    risk = state["risk_category"]
    print(f"[Router] → Routing to: {risk.lower()}_advisor")

    if risk == "CONSERVATIVE":
        return "conservative_advisor"
    elif risk == "MODERATE":
        return "moderate_advisor"
    else:
        return "aggressive_advisor"

## Nodes 2a/2b/2c — Advisor Nodes (LLM)

Only ONE of these runs per execution — determined by the router.
Each advisor specialises in a different risk profile.

In [5]:
def conservative_advisor(state: InvestorState) -> dict:
    """
    Advisor Node: Conservative (LLM)
    Recommends capital-protection instruments.
    Focus: FDs, bonds, gold, debt mutual funds, PPF.
    """
    prompt = f"""
You are a conservative investment advisor.

Investor: {state['investor_name']}, Age {state['age']}
Monthly savings available: ₹{state['monthly_savings']:,.0f}
Investment goal          : {state['investment_goal']}
Investment horizon       : {state['investment_horizon_years']} years
Risk profile             : CONSERVATIVE (score {state['risk_score']}/10)
Reason                   : {state['risk_reasoning']}

Recommend a conservative portfolio. Focus on:
  - Capital protection over returns
  - Instruments: FDs, PPF, debt mutual funds, government bonds, gold ETF
  - Avoid equity, crypto, small cap

Provide 4-5 specific instrument recommendations.
Provide an allocation breakdown that sums to 100%.
Provide a one-paragraph rationale.

Respond ONLY in raw JSON. No markdown, no extra text:
{{"recommended_instruments": ["<instrument 1>", "<instrument 2>", ...],
  "allocation_breakdown": "<e.g. 40% FD, 30% PPF, 20% Debt MF, 10% Gold ETF>",
  "advisor_rationale": "<paragraph>"}}
"""
    response = model.invoke(prompt)
    result = json.loads(response.content)
    print(f"[Conservative Advisor] → {result['allocation_breakdown']}")
    return {
        "recommended_instruments": result["recommended_instruments"],
        "allocation_breakdown": result["allocation_breakdown"],
        "advisor_rationale": result["advisor_rationale"]
    }

In [6]:
def moderate_advisor(state: InvestorState) -> dict:
    """
    Advisor Node: Moderate (LLM)
    Recommends balanced growth instruments.
    Focus: Large cap equity, balanced funds, some debt, blue chip stocks.
    """
    prompt = f"""
You are a balanced investment advisor.

Investor: {state['investor_name']}, Age {state['age']}
Monthly savings available: ₹{state['monthly_savings']:,.0f}
Investment goal          : {state['investment_goal']}
Investment horizon       : {state['investment_horizon_years']} years
Risk profile             : MODERATE (score {state['risk_score']}/10)
Reason                   : {state['risk_reasoning']}

Recommend a balanced portfolio. Focus on:
  - Mix of growth and safety
  - Instruments: Large cap equity, balanced/hybrid funds, blue chip stocks,
    some debt funds, index funds (Nifty 50)
  - Avoid crypto, small cap, derivatives

Provide 4-5 specific instrument recommendations.
Provide an allocation breakdown that sums to 100%.
Provide a one-paragraph rationale.

Respond ONLY in raw JSON. No markdown, no extra text:
{{"recommended_instruments": ["<instrument 1>", "<instrument 2>", ...],
  "allocation_breakdown": "<e.g. 50% Large Cap MF, 20% Index Fund, 20% Debt MF, 10% Gold>",
  "advisor_rationale": "<paragraph>"}}
"""
    response = model.invoke(prompt)
    result = json.loads(response.content)
    print(f"[Moderate Advisor] → {result['allocation_breakdown']}")
    return {
        "recommended_instruments": result["recommended_instruments"],
        "allocation_breakdown": result["allocation_breakdown"],
        "advisor_rationale": result["advisor_rationale"]
    }

In [7]:
def aggressive_advisor(state: InvestorState) -> dict:
    """
    Advisor Node: Aggressive (LLM)
    Recommends high-growth instruments.
    Focus: Small/mid cap, sectoral funds, direct equity, REITs.
    """
    prompt = f"""
You are a high-growth investment advisor.

Investor: {state['investor_name']}, Age {state['age']}
Monthly savings available: ₹{state['monthly_savings']:,.0f}
Investment goal          : {state['investment_goal']}
Investment horizon       : {state['investment_horizon_years']} years
Risk profile             : AGGRESSIVE (score {state['risk_score']}/10)
Reason                   : {state['risk_reasoning']}

Recommend a high-growth portfolio. Focus on:
  - Maximising long-term returns
  - Instruments: Small/mid cap funds, sectoral funds, direct equity,
    REITs, international funds, Nifty Next 50
  - Small safety buffer in liquid fund

Provide 4-5 specific instrument recommendations.
Provide an allocation breakdown that sums to 100%.
Provide a one-paragraph rationale.

Respond ONLY in raw JSON. No markdown, no extra text:
{{"recommended_instruments": ["<instrument 1>", "<instrument 2>", ...],
  "allocation_breakdown": "<e.g. 40% Small Cap MF, 30% Mid Cap MF, 20% Direct Equity, 10% Liquid Fund>",
  "advisor_rationale": "<paragraph>"}}
"""
    response = model.invoke(prompt)
    result = json.loads(response.content)
    print(f"[Aggressive Advisor] → {result['allocation_breakdown']}")
    return {
        "recommended_instruments": result["recommended_instruments"],
        "allocation_breakdown": result["allocation_breakdown"],
        "advisor_rationale": result["advisor_rationale"]
    }

## Node 3 — Report Generator (LLM)

Final node. Always runs regardless of which advisor path was taken.
Generates a professional, investor-facing report.

In [8]:
def report_generator(state: InvestorState) -> dict:
    """
    Node 3: Report Generator (LLM)
    Runs after whichever advisor was selected.
    Produces a clean, professional investor-facing report.
    """
    instruments_list = "\n  ".join(
        [f"- {i}" for i in state["recommended_instruments"]]
    )

    prompt = f"""
You are a senior wealth manager writing a formal investment report.

Investor Name      : {state['investor_name']}
Age                : {state['age']}
Investment Goal    : {state['investment_goal']}
Risk Profile       : {state['risk_category']} (score {state['risk_score']}/10)
Profile Reasoning  : {state['risk_reasoning']}

Recommended Portfolio:
  {instruments_list}

Allocation         : {state['allocation_breakdown']}
Advisor Rationale  : {state['advisor_rationale']}

Write a formal, professional investment recommendation report.
Structure it as:
  1. Investor Summary
  2. Risk Assessment
  3. Portfolio Recommendation
  4. Allocation Breakdown
  5. Important Disclaimer

Address the investor by name. Be clear, confident, professional.
Keep it concise — max 300 words.

Respond ONLY with the report text. No JSON, no markdown headers with #.
"""
    response = model.invoke(prompt)
    print(f"[Report Generator] → Report generated ({len(response.content)} chars)")
    return {"final_report": response.content}

## Build and Compile the Graph

Key difference from parallel workflow:
- `add_conditional_edges` replaces `add_edge` after profile_analyzer
- The routing function decides which single path to take
- All three advisor paths converge back to report_generator

In [9]:
graph = StateGraph(InvestorState)

# Register all nodes
graph.add_node("profile_analyzer",     profile_analyzer)
graph.add_node("conservative_advisor", conservative_advisor)
graph.add_node("moderate_advisor",     moderate_advisor)
graph.add_node("aggressive_advisor",   aggressive_advisor)
graph.add_node("report_generator",     report_generator)

# Entry point
graph.add_edge(START, "profile_analyzer")

# Conditional edge — router function decides which advisor runs
graph.add_conditional_edges(
    "profile_analyzer",     # source node
    route_by_risk,          # routing function — returns node name as string
    {
        "conservative_advisor": "conservative_advisor",
        "moderate_advisor":     "moderate_advisor",
        "aggressive_advisor":   "aggressive_advisor"
    }
)

# All three advisor paths converge to report_generator
graph.add_edge("conservative_advisor", "report_generator")
graph.add_edge("moderate_advisor",     "report_generator")
graph.add_edge("aggressive_advisor",   "report_generator")
graph.add_edge("report_generator",     END)

workflow = graph.compile()
print("Graph compiled successfully.")

Graph compiled successfully.


## Test Cases

Three test cases — one for each routing path.

In [10]:
# ── Test Case 1: Conservative ──────────────────────────────────────
conservative_investor = {
    "investor_name":            "Ramesh Iyer",
    "age":                      58,
    "monthly_income":           80000,
    "monthly_savings":          15000,
    "investment_horizon_years": 5,
    "existing_liabilities":     "Home loan EMI ₹25,000/month",
    "investment_goal":          "Retirement corpus in 5 years",
    "loss_tolerance":           "Cannot afford to lose capital",
    # initialise output fields
    "risk_category":            "",
    "risk_score":               0,
    "risk_reasoning":           "",
    "recommended_instruments":  [],
    "allocation_breakdown":     "",
    "advisor_rationale":        "",
    "final_report":             ""
}

print("=" * 70)
print(f"Processing: {conservative_investor['investor_name']}")
print("=" * 70)
result1 = workflow.invoke(conservative_investor)
print("\n── FINAL REPORT ──")
print(result1["final_report"])

Processing: Ramesh Iyer
[Profile Analyzer] → Category: CONSERVATIVE, Score: 2
[Profile Analyzer] → Reasoning: At 58 with a short investment horizon and high liabilities, Ramesh cannot afford to lose capital.
[Router] → Routing to: conservative_advisor
[Conservative Advisor] → 40% FD, 30% PPF, 20% Debt MF, 10% Gold ETF
[Report Generator] → Report generated (1729 chars)

── FINAL REPORT ──
Investor Summary  
Ramesh Iyer, aged 58, is focused on building a retirement corpus within a 5-year timeframe. Given his current life stage and financial obligations, it is crucial to prioritize capital preservation while still seeking modest growth.

Risk Assessment  
Ramesh's conservative risk profile, with a score of 2 out of 10, indicates a low tolerance for risk. At this stage in his life, he cannot afford to incur significant capital losses, making a cautious investment strategy essential.

Portfolio Recommendation  
To align with Ramesh's investment goals and risk tolerance, we recommend a diver

In [11]:
# ── Test Case 2: Moderate ──────────────────────────────────────────
moderate_investor = {
    "investor_name":            "Sneha Kulkarni",
    "age":                      35,
    "monthly_income":           120000,
    "monthly_savings":          30000,
    "investment_horizon_years": 12,
    "existing_liabilities":     "Car loan EMI ₹8,000/month",
    "investment_goal":          "Child higher education fund",
    "loss_tolerance":           "Can accept moderate fluctuation",
    "risk_category":            "",
    "risk_score":               0,
    "risk_reasoning":           "",
    "recommended_instruments":  [],
    "allocation_breakdown":     "",
    "advisor_rationale":        "",
    "final_report":             ""
}

print("=" * 70)
print(f"Processing: {moderate_investor['investor_name']}")
print("=" * 70)
result2 = workflow.invoke(moderate_investor)
print("\n── FINAL REPORT ──")
print(result2["final_report"])

Processing: Sneha Kulkarni
[Profile Analyzer] → Category: MODERATE, Score: 6
[Profile Analyzer] → Reasoning: Sneha has a medium investment horizon and can accept moderate fluctuations in her investments.
[Router] → Routing to: moderate_advisor
[Moderate Advisor] → 30% HDFC Nifty 50 Index Fund, 25% SBI Bluechip Fund, 20% ICICI Prudential Balanced Advantage Fund, 15% Axis Long Term Equity Fund, 10% HDFC Corporate Bond Fund
[Report Generator] → Report generated (1462 chars)

── FINAL REPORT ──
Investor Summary  
Sneha Kulkarni, aged 35, is focused on building a fund for her child's higher education. With a medium investment horizon, she is positioned to accept moderate fluctuations in her investments, reflecting a moderate risk profile with a score of 6 out of 10.

Risk Assessment  
Given Sneha's moderate risk tolerance, it is essential to balance growth potential with capital preservation. This approach will help mitigate the impact of market volatility while still aiming for substantial

In [12]:
# ── Test Case 3: Aggressive ────────────────────────────────────────
aggressive_investor = {
    "investor_name":            "Arjun Mehta",
    "age":                      26,
    "monthly_income":           150000,
    "monthly_savings":          60000,
    "investment_horizon_years": 20,
    "existing_liabilities":     "None",
    "investment_goal":          "Aggressive wealth creation",
    "loss_tolerance":           "Comfortable with high volatility for high returns",
    "risk_category":            "",
    "risk_score":               0,
    "risk_reasoning":           "",
    "recommended_instruments":  [],
    "allocation_breakdown":     "",
    "advisor_rationale":        "",
    "final_report":             ""
}

print("=" * 70)
print(f"Processing: {aggressive_investor['investor_name']}")
print("=" * 70)
result3 = workflow.invoke(aggressive_investor)
print("\n── FINAL REPORT ──")
print(result3["final_report"])

Processing: Arjun Mehta
[Profile Analyzer] → Category: AGGRESSIVE, Score: 10
[Profile Analyzer] → Reasoning: Arjun is young, has a long investment horizon, no liabilities, and is comfortable with high volatility for high returns.
[Router] → Routing to: aggressive_advisor
[Aggressive Advisor] → 30% SBI Small Cap Fund, 25% Axis Midcap Fund, 20% Nippon India Growth Fund, 15% HDFC Nifty Next 50 ETF, 10% Brookfield India Real Estate Trust
[Report Generator] → Report generated (1815 chars)

── FINAL REPORT ──
Investor Summary  
Arjun Mehta is a 26-year-old investor with an aggressive wealth creation goal. With a long investment horizon and no current liabilities, Arjun is well-positioned to pursue high-risk, high-reward investment opportunities.

Risk Assessment  
Arjun's risk profile is rated at 10/10, indicating a strong willingness to accept volatility in pursuit of substantial returns. This aggressive stance is supported by his youth and the absence of immediate financial obligations, al